# Multilingual BPE Tokenizer — India Wikipedia

**Assignment**: Build a shared 10,000-token BPE vocabulary for English, Hindi, Telugu, and Sanskrit  
**Score formula**: `1000 / (X_max − X_min)` where `X_i = tokens(corpus_i) / total_words(corpus_i)`

**Fertility formula**: Standard NLP definition — `total BPE tokens / total word count` (all instances).  
English target: **≤ 1.2** (instructor hard constraint).

**Optimisation strategy**:
- SCRIPT-BPE initial tokens (Unicode codepoints, not UTF-8 bytes) → no 3-byte Devanagari/Telugu penalty
- **Hybrid Pre-tokenization** → Naive whitespace split for English (to hit $\le 1.2$ constraint), LLaMA-4 style regex for Indic languages (to preserve morphology)
- **Word-Frequency Dictionary Optimization** → Performs BPE merges on unique word forms instead of the raw corpus, reducing training time from hours to seconds
- **Manual Language Multipliers** → Precisely balances the training corpus to hit the English constraint while keeping Indic fertilities competitive
- **Normal BPE** (globally most-frequent pair, Sennrich et al. 2016) → standard BPE merging

---

**Run all cells top-to-bottom.** Each step saves intermediate files so you can resume from any checkpoint.

## Step 0 — Install Dependencies

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('Dependencies installed.')

## Step 1 — Fetch Wikipedia Articles

Downloads the Wikipedia 'India' article for English, Hindi, Telugu, and Sanskrit.  
Saves cleaned plain text to `data/raw/{lang}.txt`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from fetch import fetch_all
fetch_all()

## Step 2 — Normalise Corpora

Applies Unicode NFC normalisation (all languages) and IndicNLP script normalisation (Hindi, Sanskrit, Telugu).  
Saves to `data/clean/{lang}.txt`.

In [ ]:
from normalize import normalize_all
normalize_all()

In [ ]:
# Verify corpus sizes
import os
clean_dir = os.path.join('data', 'clean')
for lang in ['en', 'hi', 'te', 'sa']:
    path = os.path.join(clean_dir, f'{lang}.txt')
    size = os.path.getsize(path)
    with open(path, encoding='utf-8') as f:
        text = f.read()
    print(f'{lang}: {len(text):>8,} chars | {len(text.split()):>8,} words | {size:>8,} bytes')

## Step 3 — Train the BPE Tokenizer

Trains the 10,000-token **Normal BPE** vocabulary with manual multipliers and dictionary optimization.  
⏱ Expected runtime: **< 1 minute** on a MacBook M-series.

Outputs:
- `output/vocab.json` — 10,000-token vocabulary
- `output/merges.txt` — ordered merge rules
- `output/tokenizer_combined.json` — widget-ready combined file
- `../widget/data/tokenizer.json` — copied for the web widget

In [ ]:
from train import train
train(target_vocab_size=10_000)

## Step 4 — Compute Fertility Ratios & Assignment Score

Loads the exported tokenizer from `output/` and scores each Wikipedia corpus independently.  
This is the **reproducibility check** — any third party can run this cell with only `vocab.json` + `merges.txt`.

In [ ]:
from score import report
results = report()

## Step 5 — Visualise Fertility Ratios

In [ ]:
import matplotlib.pyplot as plt

langs = [r['language_name'] for r in results['results']]
ratios = [r['fertility_ratio'] for r in results['results']]
colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(langs, ratios, color=colors, width=0.5)
ax.axhline(y=results['x_min'], color='gray', linestyle='--', linewidth=0.8, label=f"X_min = {results['x_min']:.4f}")
ax.axhline(y=results['x_max'], color='black', linestyle='--', linewidth=0.8, label=f"X_max = {results['x_max']:.4f}")
ax.axhline(y=1.2, color='red', linestyle=':', linewidth=1.2, label='English target = 1.2')

for bar, ratio in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ratio:.4f}', ha='center', va='bottom', fontsize=10)

score = results['score']
score_label = f"{score:.2f}" if score != float('inf') else 'MAX'
ax.set_title(f'Fertility Ratios by Language  |  Score = {score_label}', fontsize=11)
ax.set_ylabel('Fertility Ratio (tokens / total words)')
ax.set_ylim(0, max(ratios) * 1.2)
ax.legend()
plt.tight_layout()
plt.savefig('output/fertility_chart.png', dpi=150)
plt.show()
print('Chart saved to output/fertility_chart.png')

## Step 6 — Vocabulary Script Distribution

Shows how the 10,000 tokens are distributed across Unicode script blocks.

In [ ]:
import json
from tokenizer import script_distribution, load

vocab, merges = load('output')
dist = script_distribution(vocab)

print('\n=== Vocabulary Script Distribution ===')
print(f'{"Script":<14} {"Tokens":>8} {"Share":>8}')
print('-' * 34)
total = sum(dist.values())
for script, count in sorted(dist.items(), key=lambda x: -x[1]):
    pct = 100 * count / total
    print(f'{script:<14} {count:>8,} {pct:>7.1f}%')
print('-' * 34)
print(f'{"TOTAL":<14} {total:>8,} {100.0:>7.1f}%')
assert total == 10_000, f'Expected 10,000 tokens, got {total}'

In [ ]:
# Bar chart of script distribution
scripts = list(dist.keys())
counts = [dist[s] for s in scripts]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(scripts, counts, color=['#4e79a7', '#59a14f', '#f28e2b', '#bab0ac', '#e15759'][:len(scripts)])
for i, (s, c) in enumerate(zip(scripts, counts)):
    ax.text(c + 20, i, f'{c:,}  ({100*c/total:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('Number of tokens')
ax.set_title('Vocabulary Script Distribution (10,000 tokens total)')
ax.set_xlim(0, max(counts) * 1.25)
plt.tight_layout()
plt.savefig('output/script_distribution.png', dpi=150)
plt.show()

## Step 7 — Reproducibility Verification

Runs the standalone encoder using **only** `vocab.json` + `merges.txt` (no training state).  
This is exactly what a third-party grader would run.

In [ ]:
import json, os
from tokenizer import load, encode

vocab, merges = load('output')
print(f'Loaded vocab: {len(vocab)} tokens')
print(f'Loaded merges: {len(merges)} rules')

clean_dir = os.path.join('data', 'clean')
lang_info = [('en','English'), ('hi','Hindi'), ('te','Telugu'), ('sa','Sanskrit')]

print('\n=== Standalone Reproducibility Check ===')
for lang, name in lang_info:
    with open(os.path.join(clean_dir, f'{lang}.txt'), encoding='utf-8') as f:
        text = f.read()
    n_tokens = len(encode(text, vocab, merges, lang))
    # Total word count (all instances) — standard NLP fertility definition
    words = text.lower().split() if lang == 'en' else text.split()
    n_words = len(words)
    ratio = n_tokens / max(n_words, 1)
    print(f'  {name:<10}: {n_tokens:>8,} tokens / {n_words:>8,} words = {ratio:.4f}')

print('\nIf these values match Step 4 output, reproducibility is confirmed.')
print('English target: fertility ≤ 1.2')

## Summary

| Step | Output |
|------|--------|
| `data/raw/*.txt` | Raw Wikipedia articles (4 languages) |
| `data/clean/*.txt` | NFC + IndicNLP normalised corpora (scoring corpus) |
| `output/vocab.json` | 10,000-token vocabulary |
| `output/merges.txt` | Ordered BPE merge rules |
| `output/tokenizer_combined.json` | Widget-ready combined file |
| `output/fertility_chart.png` | Fertility ratio bar chart |
| `output/script_distribution.png` | Vocabulary script breakdown chart |

**Next step**: Run the widget locally with `python -m http.server 8080` from the `widget/` directory, then deploy to Netlify.